<div style="
    background-color: #0b2343; 
    color: white; 
    padding: 25px; 
    font-family: Arial, sans-serif;
">
    <h2 style="margin: 0; font-size: 24px; font-weight: normal;">
        Data Analytics & Machine Learning in Finance
    </h2>
    <h1 style="margin: 5px 0 0 0; font-size: 48px; font-weight: bold;">
        Workshop 1: Data preprocessing and Supervised Learning
    </h1>
    <div style="display: flex; justify-content: space-between; margin-top: 20px;">
        <div style="text-align: left;">
            <p style="margin: 8px 0 0 0; font-size: 18px;">
                Pedro Ramón Ventura Gómez
            </p>
            <p style="margin: 2px 0 0 0; font-size: 16px;">
                pventura@march-am.com
            </p>
        </div>
        <div style="text-align: right;">
            <p style="margin: 8px 50px 0 0; font-size: 18px;">
                Pablo Hernández Cámara
            </p>
            <p style="margin: 2px 50px 0 0; font-size: 16px;">
                pablo.hernandez-camara@uv.es
            </p>
        </div>
    </div>
</div>


# Data preprocessing and Supervised Learning

**Objectives**

- Practice cleaning and preprocessing financial data (historical fund prices).
- Extract quantitative characteristics of the assets and understand their financial meaning.

**Structure**

- Data cleaning and preprocessing: Load the historical fund price dataset, identify missing or anomalous data, and apply cleaning techniques.
- Feature engineering: Compute quantitative variables and prepare the feature matrix for clustering.

**Financial Thesis**

As portfolio managers, we believe the Asian market will perform well next year; therefore, we aim to position our fund of funds with a clear bias toward this market.

**Dataset**

- Net Asset Values (NAVs) of 25,000 investment funds between 2016-01-05 and 2021-07-16 provided by IronIA.
- Fama & French factors (Mkt-RF, SMB, HML, MOM) for Asia Pacific ex Japan.


Tenemos 25k fondos el valor liquidativo durante el periodo temporal marcado.
No sabemos que fondos a jaja......

Hay que hacer limpieza y preprocesado con sentido. Hay que quedarse con alrededor de 15k fondos.

Cartera sesgada a Asia (no solo Asia ojoooo)

Los Fama french no se donde están por los ficheros?? De ahí hay que sacar el benchmark o lo que sea...... 

## Importación

In [6]:
import pickle
import pandas as pd
import numpy as np

from dataclasses import dataclass
from __future__ import annotations

from typing import Any, Dict, Tuple

from __future__ import annotations

In [2]:
with open("../dataset/navs.pickle", "rb") as f:
    navs = pickle.load(f)

print(type(navs))

C:\Users\Miriamdbg\AppData\Local\Temp\ipykernel_3000\1136002975.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  navs = pickle.load(f)


<class 'dict'>


In [3]:
# 3. Si es dict → explorar claves
if isinstance(navs, dict):
    print(f"Número de claves: {len(navs)}")
    print("Primeras 5 claves:", list(navs.keys())[:5])

    # 4. Inspeccionar estructura interna
    first_key = list(navs.keys())[0]
    print(f"\nEjemplo clave: {first_key}")
    print("Tipo del valor:", type(navs[first_key]))

    # Si el valor es iterable (lista/array)
    try:
        print("Primeros elementos:", navs[first_key][:5])
    except Exception:
        print("No es indexable directamente")

Número de claves: 24822
Primeras 5 claves: [90, 541, 909, 915, 922]

Ejemplo clave: 90
Tipo del valor: <class 'pandas.DataFrame'>
Primeros elementos:                     isin  allfunds_id    nav                             name
date                                                                         
2016-01-05  LU0171310443           90  16.47  BGF WORLD TECHNOLOGY "A2" (EUR)
2016-01-06  LU0171310443           90  16.19  BGF WORLD TECHNOLOGY "A2" (EUR)
2016-01-07  LU0171310443           90  15.68  BGF WORLD TECHNOLOGY "A2" (EUR)
2016-01-08  LU0171310443           90  15.59  BGF WORLD TECHNOLOGY "A2" (EUR)
2016-01-11  LU0171310443           90  15.26  BGF WORLD TECHNOLOGY "A2" (EUR)


In [4]:
navs[541]

,isin,allfunds_id,nav,name
date,,,,
2016-01-05,LU0248272758,541,27.54,"BGF INDIA ""A2"""
2016-01-06,LU0248272758,541,27.50,"BGF INDIA ""A2"""
2016-01-07,LU0248272758,541,26.83,"BGF INDIA ""A2"""
2016-01-08,LU0248272758,541,27.21,"BGF INDIA ""A2"""
2016-01-11,LU0248272758,541,26.98,"BGF INDIA ""A2"""
...,...,...,...,...
2021-07-12,LU0248272758,541,45.84,"BGF INDIA ""A2"""
2021-07-13,LU0248272758,541,46.09,"BGF INDIA ""A2"""
2021-07-14,LU0248272758,541,46.10,"BGF INDIA ""A2"""


## Limpieza individual

In [8]:
START_DATE="2016-01-05"
END_DATE="2021-07-16"

Decidiremos:
* Porcentaje observaciones válidas
* Tolerancia sobre fondos no completos que mueren o nacen entre las fechas deseadas
* Gap máximo entre observacioens consecutivas
* Mínimo absoluto de observaciones

In [15]:
@dataclass(frozen=True)
class CleaningConfig:
    start_date: str = START_DATE
    end_date: str = END_DATE

    # Cobertura mínima sobre el número esperado de observaciones
    min_coverage_daily: float = 0.85
    min_coverage_weekly: float = 0.75
    min_coverage_monthly: float = 0.65

    # Tolerancia máxima entre el inicio/fin de ventana y la primera/última obs
    max_start_gap_daily_days: int = 5
    max_start_gap_weekly_days: int = 14
    max_start_gap_monthly_days: int = 45

    max_end_gap_daily_days: int = 5
    max_end_gap_weekly_days: int = 14
    max_end_gap_monthly_days: int = 45

    # Gap máximo interno permitido entre observaciones consecutivas
    max_internal_gap_daily_days: int = 5
    max_internal_gap_weekly_days: int = 14
    max_internal_gap_monthly_days: int = 62

    # Máximo número de gaps grandes permitidos
    max_n_large_gaps_daily: int = 0
    max_n_large_gaps_weekly: int = 1
    max_n_large_gaps_monthly: int = 1

    # Mínimo absoluto de observaciones
    min_obs_daily: int = 200
    min_obs_weekly: int = 80
    min_obs_monthly: int = 24

    # Si True, elimina fondos que no cubran razonablemente todo el intervalo
    require_full_window: bool = True

    # Lógica anti-series raras / mixtas
    min_share_daily_like: float = 0.70
    min_share_weekly_like: float = 0.70
    min_share_monthly_like: float = 0.50

    max_share_other_gaps_daily: float = 0.20
    max_share_other_gaps_weekly: float = 0.25
    max_share_other_gaps_monthly: float = 0.30

    min_dominant_share: float = 0.75

    # Control de stale prices / NAV plano
    max_flat_streak_daily: int = 10
    max_flat_streak_weekly: int = 4
    max_flat_streak_monthly: int = 3

    # Control de retornos extremos
    max_abs_return_daily: float = 0.50
    max_abs_return_weekly: float = 0.75
    max_abs_return_monthly: float = 1.00

    # Si True, rechaza series con retornos extremos
    reject_extreme_returns: bool = True


def clean_single_fund(
    df: pd.DataFrame,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
) -> pd.DataFrame:
    """Limpia una serie individual y la recorta a la ventana de análisis."""
    out = df.copy()

    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index, errors="coerce")

    out = out.loc[~out.index.isna()]
    out = out.sort_index()
    out = out.loc[~out.index.duplicated(keep="last")]

    if "nav" not in out.columns:
        return pd.DataFrame(columns=out.columns)

    out["nav"] = pd.to_numeric(out["nav"], errors="coerce")
    out = out.dropna(subset=["nav"])
    out = out.loc[out["nav"] > 0]

    out = out.loc[(out.index >= start_date) & (out.index <= end_date)]

    return out


def infer_periodicity(
    index: pd.DatetimeIndex,
    cfg: CleaningConfig,
) -> tuple[str, dict[str, float | int]]:
    """
    Infiere periodicidad y detecta series irregulares a partir de la
    distribución de gaps entre observaciones.
    """
    diagnostics: dict[str, float | int] = {
        "median_gap_days": np.nan,
        "mean_gap_days": np.nan,
        "p90_gap_days": np.nan,
        "share_daily_like": np.nan,
        "share_weekly_like": np.nan,
        "share_monthly_like": np.nan,
        "share_other_gaps": np.nan,
        "dominant_share": np.nan,
    }

    if len(index) < 3:
        return "unknown", diagnostics

    deltas = np.diff(index.values).astype("timedelta64[D]").astype(int)
    deltas = deltas[deltas > 0]

    if len(deltas) == 0:
        return "unknown", diagnostics

    median_gap = float(np.median(deltas))
    mean_gap = float(np.mean(deltas))
    p90_gap = float(np.percentile(deltas, 90))

    daily_like = deltas <= 3
    weekly_like = (deltas >= 4) & (deltas <= 10)
    monthly_like = (deltas >= 20) & (deltas <= 40)

    other_gaps = ~(daily_like | weekly_like | monthly_like)

    share_daily_like = float(np.mean(daily_like))
    share_weekly_like = float(np.mean(weekly_like))
    share_monthly_like = float(np.mean(monthly_like))
    share_other_gaps = float(np.mean(other_gaps))

    dominant_share = max(
        share_daily_like,
        share_weekly_like,
        share_monthly_like,
    )

    diagnostics = {
        "median_gap_days": median_gap,
        "mean_gap_days": mean_gap,
        "p90_gap_days": p90_gap,
        "share_daily_like": share_daily_like,
        "share_weekly_like": share_weekly_like,
        "share_monthly_like": share_monthly_like,
        "share_other_gaps": share_other_gaps,
        "dominant_share": dominant_share,
    }

    if dominant_share < cfg.min_dominant_share:
        return "irregular", diagnostics

    if (
        share_daily_like >= cfg.min_share_daily_like
        and share_other_gaps <= cfg.max_share_other_gaps_daily
        and median_gap <= 3
    ):
        return "daily", diagnostics

    if (
        share_weekly_like >= cfg.min_share_weekly_like
        and share_other_gaps <= cfg.max_share_other_gaps_weekly
        and 4 <= median_gap <= 10
    ):
        return "weekly", diagnostics

    if (
        share_monthly_like >= cfg.min_share_monthly_like
        and share_other_gaps <= cfg.max_share_other_gaps_monthly
        and median_gap >= 20
    ):
        return "monthly", diagnostics

    return "irregular", diagnostics


def expected_obs_count(
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    periodicity: str,
) -> int:
    """Número esperado aproximado de observaciones en la ventana."""
    if periodicity == "daily":
        return len(pd.date_range(start=start_date, end=end_date, freq="B"))
    if periodicity == "weekly":
        return len(pd.date_range(start=start_date, end=end_date, freq="W-FRI"))
    if periodicity == "monthly":
        return len(pd.date_range(start=start_date, end=end_date, freq="BME"))
    return 0


def get_thresholds(
    periodicity: str,
    cfg: CleaningConfig,
) -> dict[str, float | int]:
    """Devuelve thresholds según periodicidad."""
    if periodicity == "daily":
        return {
            "min_coverage": cfg.min_coverage_daily,
            "max_start_gap": cfg.max_start_gap_daily_days,
            "max_end_gap": cfg.max_end_gap_daily_days,
            "max_internal_gap": cfg.max_internal_gap_daily_days,
            "max_n_large_gaps": cfg.max_n_large_gaps_daily,
            "min_obs": cfg.min_obs_daily,
            "max_flat_streak": cfg.max_flat_streak_daily,
            "max_abs_return": cfg.max_abs_return_daily,
        }

    if periodicity == "weekly":
        return {
            "min_coverage": cfg.min_coverage_weekly,
            "max_start_gap": cfg.max_start_gap_weekly_days,
            "max_end_gap": cfg.max_end_gap_weekly_days,
            "max_internal_gap": cfg.max_internal_gap_weekly_days,
            "max_n_large_gaps": cfg.max_n_large_gaps_weekly,
            "min_obs": cfg.min_obs_weekly,
            "max_flat_streak": cfg.max_flat_streak_weekly,
            "max_abs_return": cfg.max_abs_return_weekly,
        }

    if periodicity == "monthly":
        return {
            "min_coverage": cfg.min_coverage_monthly,
            "max_start_gap": cfg.max_start_gap_monthly_days,
            "max_end_gap": cfg.max_end_gap_monthly_days,
            "max_internal_gap": cfg.max_internal_gap_monthly_days,
            "max_n_large_gaps": cfg.max_n_large_gaps_monthly,
            "min_obs": cfg.min_obs_monthly,
            "max_flat_streak": cfg.max_flat_streak_monthly,
            "max_abs_return": cfg.max_abs_return_monthly,
        }

    return {
        "min_coverage": 1.0,
        "max_start_gap": 0,
        "max_end_gap": 0,
        "max_internal_gap": 0,
        "max_n_large_gaps": 0,
        "min_obs": 999999,
        "max_flat_streak": 0,
        "max_abs_return": 0.0,
    }


def max_flat_streak(nav: pd.Series) -> int:
    """Máxima racha de NAV sin cambio respecto al dato anterior."""
    if nav.empty:
        return 0

    same_as_prev = nav.eq(nav.shift(1))
    if same_as_prev.sum() == 0:
        return 0

    group_id = same_as_prev.ne(same_as_prev.shift()).cumsum()
    streaks = same_as_prev.groupby(group_id).sum()

    return int(streaks.max()) if len(streaks) > 0 else 0


def validate_fund(
    fund_id: int,
    df: pd.DataFrame,
    cfg: CleaningConfig,
) -> dict[str, Any]:
    """Valida un fondo y devuelve métricas + motivo de aceptación/rechazo."""
    start_date = pd.Timestamp(cfg.start_date)
    end_date = pd.Timestamp(cfg.end_date)

    cleaned = clean_single_fund(df=df, start_date=start_date, end_date=end_date)

    base_info = {
        "allfunds_id": fund_id,
        "isin": None,
        "name": None,
        "periodicity": "unknown",
        "n_obs": 0,
        "expected_obs": 0,
        "coverage_ratio": 0.0,
        "first_date": pd.NaT,
        "last_date": pd.NaT,
        "start_gap_days": np.nan,
        "end_gap_days": np.nan,
        "max_internal_gap_days": np.nan,
        "n_large_gaps": np.nan,
        "median_gap_days": np.nan,
        "mean_gap_days": np.nan,
        "p90_gap_days": np.nan,
        "share_daily_like": np.nan,
        "share_weekly_like": np.nan,
        "share_monthly_like": np.nan,
        "share_other_gaps": np.nan,
        "dominant_share": np.nan,
        "max_flat_streak": np.nan,
        "max_abs_return": np.nan,
        "is_valid": False,
        "reject_reason": "",
    }

    if cleaned.empty:
        base_info["reject_reason"] = "sin_datos_en_ventana"
        return base_info

    if "isin" in cleaned.columns and not cleaned["isin"].empty:
        base_info["isin"] = cleaned["isin"].iloc[0]

    if "name" in cleaned.columns and not cleaned["name"].empty:
        base_info["name"] = cleaned["name"].iloc[0]

    periodicity, diagnostics = infer_periodicity(cleaned.index, cfg=cfg)
    thresholds = get_thresholds(periodicity=periodicity, cfg=cfg)

    first_date = cleaned.index.min()
    last_date = cleaned.index.max()
    n_obs = len(cleaned)

    expected_obs = expected_obs_count(
        start_date=start_date,
        end_date=end_date,
        periodicity=periodicity,
    )

    coverage_ratio = n_obs / expected_obs if expected_obs > 0 else 0.0

    deltas = np.diff(cleaned.index.values).astype("timedelta64[D]").astype(int)
    deltas = deltas[deltas > 0]

    max_internal_gap_days = int(deltas.max()) if len(deltas) > 0 else 0
    n_large_gaps = int(np.sum(deltas > int(thresholds["max_internal_gap"])))

    start_gap_days = int((first_date - start_date).days)
    end_gap_days = int((end_date - last_date).days)

    flat_streak = max_flat_streak(cleaned["nav"])

    returns = cleaned["nav"].pct_change()
    max_abs_return = (
        float(returns.abs().max())
        if returns.notna().any()
        else np.nan
    )

    base_info.update(
        {
            "periodicity": periodicity,
            "n_obs": n_obs,
            "expected_obs": expected_obs,
            "coverage_ratio": coverage_ratio,
            "first_date": first_date,
            "last_date": last_date,
            "start_gap_days": start_gap_days,
            "end_gap_days": end_gap_days,
            "max_internal_gap_days": max_internal_gap_days,
            "n_large_gaps": n_large_gaps,
            "median_gap_days": diagnostics["median_gap_days"],
            "mean_gap_days": diagnostics["mean_gap_days"],
            "p90_gap_days": diagnostics["p90_gap_days"],
            "share_daily_like": diagnostics["share_daily_like"],
            "share_weekly_like": diagnostics["share_weekly_like"],
            "share_monthly_like": diagnostics["share_monthly_like"],
            "share_other_gaps": diagnostics["share_other_gaps"],
            "dominant_share": diagnostics["dominant_share"],
            "max_flat_streak": flat_streak,
            "max_abs_return": max_abs_return,
        }
    )

    if periodicity == "unknown":
        base_info["reject_reason"] = "periodicidad_desconocida"
        return base_info

    if periodicity == "irregular":
        base_info["reject_reason"] = "serie_irregular"
        return base_info

    if n_obs < int(thresholds["min_obs"]):
        base_info["reject_reason"] = "muy_pocas_observaciones"
        return base_info

    if coverage_ratio < float(thresholds["min_coverage"]):
        base_info["reject_reason"] = "cobertura_insuficiente"
        return base_info

    if cfg.require_full_window and start_gap_days > int(thresholds["max_start_gap"]):
        base_info["reject_reason"] = "empieza_demasiado_tarde"
        return base_info

    if cfg.require_full_window and end_gap_days > int(thresholds["max_end_gap"]):
        base_info["reject_reason"] = "termina_demasiado_pronto"
        return base_info

    if max_internal_gap_days > int(thresholds["max_internal_gap"]):
        base_info["reject_reason"] = "gaps_internos_excesivos"
        return base_info

    if n_large_gaps > int(thresholds["max_n_large_gaps"]):
        base_info["reject_reason"] = "demasiados_gaps_grandes"
        return base_info

    if flat_streak > int(thresholds["max_flat_streak"]):
        base_info["reject_reason"] = "nav_demasiado_plano"
        return base_info

    if (
        cfg.reject_extreme_returns
        and pd.notna(max_abs_return)
        and max_abs_return > float(thresholds["max_abs_return"])
    ):
        base_info["reject_reason"] = "retorno_extremo"
        return base_info

    base_info["is_valid"] = True
    base_info["reject_reason"] = ""

    return base_info


def filter_navs_dictionary(
    navs: Dict[int, pd.DataFrame],
    cfg: CleaningConfig,
) -> Tuple[Dict[int, pd.DataFrame], pd.DataFrame]:
    """
    Devuelve:
    - diccionario limpio
    - tabla resumen/auditoría
    """
    start_date = pd.Timestamp(cfg.start_date)
    end_date = pd.Timestamp(cfg.end_date)

    clean_navs: Dict[int, pd.DataFrame] = {}
    audit_rows: list[dict[str, Any]] = []

    for fund_id, df in navs.items():
        audit = validate_fund(fund_id=fund_id, df=df, cfg=cfg)
        audit_rows.append(audit)

        if audit["is_valid"]:
            clean_df = clean_single_fund(
                df=df,
                start_date=start_date,
                end_date=end_date,
            )
            clean_navs[fund_id] = clean_df

    audit_df = pd.DataFrame(audit_rows).sort_values(
        by=["is_valid", "periodicity", "coverage_ratio", "n_obs"],
        ascending=[False, True, False, False],
    )

    return clean_navs, audit_df



In [16]:

# =========================
# EJECUCIÓN
# =========================

cfg = CleaningConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    min_coverage_daily=0.85,
    min_coverage_weekly=0.75,
    min_coverage_monthly=0.65,
    max_start_gap_daily_days=5,
    max_start_gap_weekly_days=14,
    max_start_gap_monthly_days=45,
    max_end_gap_daily_days=5,
    max_end_gap_weekly_days=14,
    max_end_gap_monthly_days=45,
    max_internal_gap_daily_days=5,
    max_internal_gap_weekly_days=14,
    max_internal_gap_monthly_days=62,
    max_n_large_gaps_daily=0,
    max_n_large_gaps_weekly=1,
    max_n_large_gaps_monthly=1,
    min_obs_daily=200,
    min_obs_weekly=80,
    min_obs_monthly=24,
    require_full_window=True,
    min_share_daily_like=0.70,
    min_share_weekly_like=0.70,
    min_share_monthly_like=0.50,
    max_share_other_gaps_daily=0.20,
    max_share_other_gaps_weekly=0.25,
    max_share_other_gaps_monthly=0.30,
    min_dominant_share=0.75,
    max_flat_streak_daily=10,
    max_flat_streak_weekly=4,
    max_flat_streak_monthly=3,
    max_abs_return_daily=0.50,
    max_abs_return_weekly=0.75,
    max_abs_return_monthly=1.00,
    reject_extreme_returns=True,
)

sample_ids = list(navs.keys())[:5]

sample_navs = {k: navs[k] for k in sample_ids}


navs_clean, audit_df = filter_navs_dictionary(navs=sample_navs, cfg=cfg)

print(f"Fondos originales: {len(sample_navs):,}")
print(f"Fondos válidos: {len(navs_clean):,}")
print(f"Fondos rechazados: {len(sample_navs) - len(navs_clean):,}")

print("\nResumen por periodicidad y validez:")
print(audit_df.groupby(["periodicity", "is_valid"]).size())

print("\nMotivos de rechazo:")
print(
    audit_df.loc[~audit_df["is_valid"], "reject_reason"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nPrimeras filas de auditoría:")
print(audit_df.head())


# =========================
# INSPECCIONES ÚTILES
# =========================

# 1. Ver solo fondos válidos
audit_valid = audit_df.loc[audit_df["is_valid"]].copy()

# 2. Ver solo rechazados
audit_rejected = audit_df.loc[~audit_df["is_valid"]].copy()

# 3. Guardar auditoría
# audit_df.to_csv("audit_navs_cleaning.csv", index=False)

# 4. Ejemplos de fondos irregulares
irregular_examples = audit_df.loc[
    audit_df["reject_reason"] == "serie_irregular",
    [
        "allfunds_id",
        "name",
        "periodicity",
        "median_gap_days",
        "p90_gap_days",
        "share_daily_like",
        "share_weekly_like",
        "share_monthly_like",
        "share_other_gaps",
        "dominant_share",
    ],
].head(20)

print("\nEjemplos de series irregulares:")
print(irregular_examples)

# 5. Ejemplos de NAV demasiado plano
flat_examples = audit_df.loc[
    audit_df["reject_reason"] == "nav_demasiado_plano",
    [
        "allfunds_id",
        "name",
        "periodicity",
        "max_flat_streak",
        "coverage_ratio",
        "n_obs",
    ],
].head(20)

print("\nEjemplos de NAV demasiado plano:")
print(flat_examples)

# 6. Ejemplos de retornos extremos
extreme_examples = audit_df.loc[
    audit_df["reject_reason"] == "retorno_extremo",
    [
        "allfunds_id",
        "name",
        "periodicity",
        "max_abs_return",
        "coverage_ratio",
        "n_obs",
    ],
].head(20)

print("\nEjemplos de retorno extremo:")
print(extreme_examples)

Fondos originales: 5
Fondos válidos: 0
Fondos rechazados: 5

Resumen por periodicidad y validez:
periodicity  is_valid
daily        False       5
dtype: int64

Motivos de rechazo:
reject_reason
empieza_demasiado_tarde    3
gaps_internos_excesivos    2
Name: count, dtype: int64

Primeras filas de auditoría:
   allfunds_id          isin                                    name  \
0           90  LU0171310443         BGF WORLD TECHNOLOGY "A2" (EUR)   
1          541  LU0248272758                          BGF INDIA "A2"   
2          909  LU1408527916    BGF EMERGING MARKETS  "A" (GBPHDG) D   
3          915  LU1408528211       BGF EMERGING MARKETS  "A" (USD) D   
4          922  LU1408528302  BGF EMERGING MARKETS  "A" (GBPHDG) D A   

  periodicity  n_obs  expected_obs  coverage_ratio first_date  last_date  \
0       daily   1387          1444        0.960526 2016-01-05 2021-07-16   
1       daily   1373          1444        0.950831 2016-01-05 2021-07-16   
2       daily   1299          1